### 문서 로딩 & 청킹
- pdf 를 langchain 로더를 활용해서 문서 형태로 변화
- 정리한 페이지를 csv 로 저장해서 -> 재사용도 가능한 형태로 바꿔도 봅시다.

### 문서를 통째로(268) 한꺼번에 넣을경우 생기는 문제
1. 임베딩 모델 입력한도
    - small -> 8000 정도 토큰이 최대
    - 나눠서 넣어야
2. 한 문서를 통째로 정도면 주제가 섞여 있을 때
    - 벡터화 하더라도 주제에 따라서 분류하기가 적절하지 않다.
    - 한 페이지에 여러 주제가 있다면 -> 벡터화 했을때 어디로?
    예) 한 페이지에 1. 청년 월세 지원  2. 노인 복지 지원 주제가 있다면
        - 분리해서 벡터화 하는게 맞다


In [3]:
from pypdf import PdfReader

reader = PdfReader("../data/16-1_K희망사다리2026_모두의정책.pdf")
total_k_ladder = len(reader.pages)
total_k_ladder

268

In [4]:
reader.pages[0]

{'/ArtBox': [0.0, 0.0, 430.866, 632.126],
 '/BleedBox': [0.0, 0.0, 430.866, 632.126],
 '/Contents': {'/Filter': '/FlateDecode'},
 '/CropBox': [0.0, 0.0, 430.866, 632.126],
 '/MediaBox': [0.0, 0.0, 430.866, 632.126],
 '/Parent': {'/Count': 4,
  '/Kids': [IndirectObject(1, 0, 1812423497472),
   IndirectObject(27, 0, 1812423497472),
   IndirectObject(29, 0, 1812423497472),
   IndirectObject(26110, 0, 1812423497472)],
  '/Parent': {'/Count': 34,
   '/Kids': [IndirectObject(26109, 0, 1812423497472),
    IndirectObject(26111, 0, 1812423497472),
    IndirectObject(26117, 0, 1812423497472),
    IndirectObject(26123, 0, 1812423497472),
    IndirectObject(26129, 0, 1812423497472),
    IndirectObject(26135, 0, 1812423497472),
    IndirectObject(26141, 0, 1812423497472)],
   '/Parent': {'/Count': 268,
    '/Kids': [IndirectObject(26108, 0, 1812423497472),
     IndirectObject(26147, 0, 1812423497472),
     IndirectObject(26178, 0, 1812423497472),
     IndirectObject(26240, 0, 1812423497472),
     I

In [8]:
# 내용만 살짝 확인
# for i, page in enumerate(reader.pages[:5]):
#     print()

text = reader.pages[4].extract_text()
print(text)


아동·청소년
060
061
062
063
064
065
066
067
068
069
070
071
072
073
074
075
076
077
078
080
첫만남 이용권
3~5세 유치원 학비
산모·신생아 건강관리
영유아 건강검진
초·중·고 학생 교육정보화 지원(PC, 인터넷 통신비)
온동네 초등돌봄·교육
가족돌봄휴가
지역 아동센터
저소득 한부모가족 아동양육비
그 밖의 연장형 보육료 지원
가족돌봄휴직
가정 밖 청소년 지원
학교 밖 청소년 지원
복권기금 꿈사다리 장학사업
드림장학금(우수고등학생 해외유학장학금)
고립·은둔 청소년 원스톱 패키지 지원
학교 밖 청소년 자립·취업 지원
영유아보육료
인구감소지역 청소년 성장지원
방과후 보육료 지원(12세 이하 초등학교 취학아동)
어르신
102
103
104
105
106
107
108
110
노인 장기요양 시설·재가서비스
보훈 재가복지서비스 지원(노인)
의료급여수급자 노인 틀니 지원
치매안심센터
치매상담 콜센터
노인맞춤 돌봄서비스
농지이양 은퇴 직불사업
고령 운전자 교통안전교육
청년·대학생
082
083
084
086
087
088
089
090
092
097
098
100
국가장학금 지원(대학생)
한국형 온라인 공개강좌(K-MOOC)
일반상환 학자금 대출
햇살론유스
해외취업지원(K-Move 스쿨)
농촌출신대학생 학자금 대출
청년창업사관학교
주거안정장학금
청년 일자리
청년 사회·복지
청년 주거지원
청년 자산형성
가족·여성
112
113
114
115
116
117
118
119
120
121
122
124
125
126
127
128
129
130
131
132
133
가정양육수당
국민행복카드
취약계층 아동통합서비스
(드림스타트)
아이돌봄 서비스
여성창업보육센터
여성새로일하기센터
다함께 돌봄센터
임산부 및 영유아 영양플러스
육아휴직급여+
육아기 근로시간 단축급여
맘편한 임신 원스톱 서비스
행복출산 원스톱 서비스
건강보험 임신·출산 진료비 지원
고용보험 미적용자 출산급여 지원
여성가장 창업자금지원
부모급여
한부모가족 복지시설

In [10]:
from dotenv import load_dotenv                                              # dotenv: .env 파일에서 환경 변수(API Key 등)를 로드하는 라이브러리
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

load_dotenv()                                                               # .env 파일에 등록된 환경 변수 불러오기

True

### 로드맵
1. 문서 로드
2. 문서 분할
3. 크로마 DB 적재
4. RAG 구축

In [11]:
pdf_docs = []
for i,  page in enumerate(reader.pages):
    text = page.extract_text()
    pdf_docs.append(Document(page_content=text,
                             metadata={"source":"K희망사다리2026_모두의정책",
                                       "page": i + 1 }))

pdf_docs[:10]    

[Document(metadata={'source': 'K희망사다리2026_모두의정책', 'page': 1}, page_content='발간등록번호\n11-1371000-100161-01\nK-희망사다리\n모두의 정책\n2026\n국민생활지원 정보 모음집\n●2026년 신규 민생지원 제도  ●지방우대 패키지 \n●내게 필요한 서비스 키워드로 바로 찾기\n●신청해야 받는 숨은 정부지원금 찾기'),
 Document(metadata={'source': 'K희망사다리2026_모두의정책', 'page': 2}, page_content=''),
 Document(metadata={'source': 'K희망사다리2026_모두의정책', 'page': 3}, page_content='K-희망사다리\n모두의 정책\n2026'),
 Document(metadata={'source': 'K희망사다리2026_모두의정책', 'page': 4}, page_content='모두의 정책 \nK-희망사다리 2026차\n례\n2026년 신규 민생지원 제도\n008\n009\n010\n011\n012\n013\n014\n015\n016\n018\n유아 단계적 무상교육·보육\n참전유공자 등 생계지원금 지급\n새도약기금\n새도약론\n청년미래적금\n장기간부 도약적금\n범죄피해구조금 확대\n범죄피해자 긴급 생활안정비\n중소기업 직장인 든든한 한 끼\n보호대상아동 민간후원 장학사업\n지방우대 패키지\n020\n021\n022\n023\n024\n025\n026\n028\n030\n농어촌 기본소득 시범사업\n고령자 계속고용 장려금 \n비수도권기업 지원 확대\n노인 일자리 및 사회활동 지원\n국민내일배움카드\n지역사랑상품권\n창업사업화지원(초기·도약패키지)\n중소기업 혁신바우처 지원\n팁스(TIPS)\n청년 일자리 도약장려금 비수도권 \n우대지원\n숨은 정부지원금 찾기\n032\n033\n034\n035\n036\n037\n038\n039\n040\n042\n043\n044\n045\n046\n048\n0

### 문서 청킹

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=400,
                                          chunk_overlap=80
                                          )

chunks = splitter.split_documents(pdf_docs)
len(chunks)

606

In [14]:
print(chunks[20].page_content)

새도약기금
 1660-0705
새도약기금
010따뜻한 동행 모두가 행복한 사회 - 2026년 신규 민생지원 제도
지원대상 	 •	 정책발표일(2025년	6월	19일)	기준	①	금융회사별로	개인(개인사업자	포함)이	
②	7년	이상	연체	중인	무담보	채무	계좌의	원금	합계액이	③	5,000만	원	이하
인	경우
핵심내용 	 • 	대상채권	일괄매입	후	즉시	추심을	중단하며,	행정	데이터	등을	활용한	상환
능력	심사를	거쳐	채무자의	여건(상환능력	등)에	따라	소각	또는	채무조정	
지원	
	 	 -	상환능력	없음:	소각(최대	5,000만	원)
	 	 -	상환능력	있음:	강화된	채무조정(최대	80%,	최장	10년	분할상환)
	 •	2025년	10월	1일	새도약기금	출범	후	1년간	대상채권	일괄매입	및	순차적


In [16]:
# 임베딩 및 저장
DB_PATH = "../data/k_ladder_2026"


# 임베딩 모델 설정
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 벡터스토어에 저장
vectorstore = Chroma.from_documents(
    documents=pdf_docs,
    embedding=embeddings,
    collection_name="k_ladder_2026",
    persist_directory=DB_PATH
)

In [17]:
reg_docs =  vectorstore.similarity_search("청년 지월 월세는 어떻게 신청하나요?",k=10)
reg_docs

[Document(id='66385e0d-284f-43c8-93ca-5f24b39b01bb', metadata={'page': 14, 'source': 'K희망사다리2026_모두의정책'}, page_content='청년미래적금\n 1600-5500\n금융위원회\n012따뜻한 동행 모두가 행복한 사회 - 2026년 신규 민생지원 제도\n지원대상 \t •\t 일정\t소득\t이하\t만\t19~34세\t청년(병역\t최대\t6년\t인정)\n\t \t - \t\t일반형:\t개인\t소득\t6,000만\t원\t이하\t소득자\t또는\t연\t매출\t3억\t원\t이하\t소상공인\t\n중\t가구\t중위소득\t200%\t이하\n\t \t - \t\t우대형:\t개인소득\t3,600만\t원\t이하\t중소기업\t재직자\t또는\t연\t매출\t1억\t원\t\n이하\t소상공인\t중\t가구\t중위소득\t150%\t이하\n   ※  일반형 요건을 충족하는 중소기업 신규 재직자는 우대형 분류\n핵심내용 \t • \t만기\t3년\n\t •\t납입액(월\t50만\t원\t한도)에\t대한\t정부기여금\t지원(일반형\t6%,\t우대형\t12%)\t\n및\t이자소득\t비과세\n  ※  개인소득 6,000~7,500만 원 이하는 이자소득 비과세만 부여\n이용방법 \t • \t신청\t기간:\t2026년\t6월\t이후(추후\t안내\t예정)\n\t •신청\t방법:\t비대면\t가입\t신청(추후\t안내\t예정)\n문의처\t •금융위원회(☎1600-5500) \t및\t서민금융진흥원\n최대 2,000만 원 \n이상\n3년\t만기'),
 Document(id='a7734810-02d3-40d0-9151-b5be764009ba', metadata={'source': 'K희망사다리2026_모두의정책', 'page': 100}, page_content='098생애주기별 국민생활 서비스 - 청년·대학생\n청년 주거지원 \n 1600-1004\nLH청약플러스\n사업명 지원대상 및 핵심내용 시기 이용방법(문의처)\n다가구 \n매입 임대

In [20]:
print(reg_docs[0].page_content)

청년미래적금
 1600-5500
금융위원회
012따뜻한 동행 모두가 행복한 사회 - 2026년 신규 민생지원 제도
지원대상 	 •	 일정	소득	이하	만	19~34세	청년(병역	최대	6년	인정)
	 	 - 		일반형:	개인	소득	6,000만	원	이하	소득자	또는	연	매출	3억	원	이하	소상공인	
중	가구	중위소득	200%	이하
	 	 - 		우대형:	개인소득	3,600만	원	이하	중소기업	재직자	또는	연	매출	1억	원	
이하	소상공인	중	가구	중위소득	150%	이하
   ※  일반형 요건을 충족하는 중소기업 신규 재직자는 우대형 분류
핵심내용 	 • 	만기	3년
	 •	납입액(월	50만	원	한도)에	대한	정부기여금	지원(일반형	6%,	우대형	12%)	
및	이자소득	비과세
  ※  개인소득 6,000~7,500만 원 이하는 이자소득 비과세만 부여
이용방법 	 • 	신청	기간:	2026년	6월	이후(추후	안내	예정)
	 •신청	방법:	비대면	가입	신청(추후	안내	예정)
문의처	 •금융위원회(☎1600-5500) 	및	서민금융진흥원
최대 2,000만 원 
이상
3년	만기
